<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 24 · Machine Learning Workflow for Fixed Income

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch24_machine_learning_workflow_fixed_income.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

This lab connects Chapter 24 to a reproducible calculation. You will inspect the chapter's inputs, reproduce a core machine learning workflow for fixed income relationship, and test how a controlled change affects the result.

By the end, you should be able to identify the model inputs and units, verify one numerical output, and state the limitation that matters before interpreting it.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The next cell runs the transparent Python implementation and prints its reproducible summary. Read the inputs and units in the script alongside the output; the printed values are teaching examples, not investment recommendations.

In [ ]:
SCRIPT = ROOT / "code" / "ch24_machine_learning_workflow_fixed_income.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Use the script output as controlled evidence. Check the units, sign convention, source dataset, and the economic direction of the result before using it in a written explanation or GenAI-assisted summary.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch24_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Make the time ordering explicit and build a model-card example from the separate illustrative metric snapshot. Its numbers are not the fitted companion script's holdout results.

In [ ]:
panel = pd.read_csv(DATA / "ch24_ml_feature_panel.csv", parse_dates=["date"])
comparison = pd.read_csv(DATA / "ch24_model_comparison.csv")
chronological = panel.date.is_monotonic_increasing
model_card = pd.DataFrame({
    "business_question": ["one-step curve target"],
    "target_horizon": ["next observation"],
    "split_rule": ["chronological"],
    "baseline": [comparison.model.iloc[0]],
    "illustrative_lowest_mae": [
        comparison.sort_values("mae").model.iloc[0]
    ],
    "dates_in_order": [chronological],
})
model_card

## Mini-lab: Inspect time ordering and the benchmark

The stored comparison is an independent illustrative metric snapshot, not the output of the chapter's fitted model. Use the preceding script output when discussing that model's holdout performance.

In [ ]:
import pandas as pd

panel = pd.read_csv(DATA / "ch24_ml_feature_panel.csv",
                    parse_dates=["date"])
comparison = pd.read_csv(DATA / "ch24_model_comparison.csv")
pd.Series({"chronological": panel.date.is_monotonic_increasing,
           "rows": len(panel)}), comparison


## Exercises

Use the displayed inputs and the chapter script to complete these checks. Record units and sign conventions with every result.

1. Verify that the train/test split is chronological.
2. Compare the model MAE with the no-change baseline.
3. Explain why the target horizon and label end date matter.

For more detail, see Chapter 24 of *Python & AI for Rates, Bonds, and Credit* and its referenced sections on the topic.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">